In [47]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from primer.config import ModelConfig
from primer.model import Transformer


In [49]:
ref_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

In [50]:
# Qwen3Config {
#   "architectures": [
#     "Qwen3ForCausalLM"
#   ],
#   "attention_bias": false,
#   "attention_dropout": 0.0,
#   "bos_token_id": 151643,
#   "eos_token_id": 151645,
#   "head_dim": 128,
#   "hidden_act": "silu",
#   "hidden_size": 1024,
#   "initializer_range": 0.02,
#   "intermediate_size": 3072,
#   "max_position_embeddings": 40960,
#   "max_window_layers": 28,
#   "model_type": "qwen3",
#   "num_attention_heads": 16,
#   "num_hidden_layers": 28,
#   "num_key_value_heads": 8,
#   "rms_norm_eps": 1e-06,
#   "rope_scaling": null,
#   "rope_theta": 1000000,
#   "sliding_window": null,
#   "tie_word_embeddings": true,
#   "torch_dtype": "float32",
#   "transformers_version": "4.53.1",
#   "use_cache": true,
#   "use_sliding_window": false,
#   "vocab_size": 151936
# }

In [51]:
config = ModelConfig(
    d_model=ref_model.config.hidden_size,
    n_layers=ref_model.config.num_hidden_layers,
    max_seqlen=ref_model.config.max_position_embeddings,
    vocab_size=ref_model.config.vocab_size,
    eos_id=ref_model.config.eos_token_id,
    tie_embeddings=ref_model.config.tie_word_embeddings,
    parallel_layers=False,
    multiple_of=1,
    # ================    
    # Attention Config
    # ================    
    n_heads=ref_model.config.num_attention_heads,
    n_kv_heads=ref_model.config.num_key_value_heads,
    head_dim=ref_model.config.head_dim,
    dropout_p=ref_model.config.attention_dropout,
    attn_bias=ref_model.config.attention_bias,
    attn_prenorm=True,
    attn_postnorm=False,
    qknorm=True,
    qknorm_use_global=False,
    # ==================    
    # FeedForward Config
    # ==================
    intermediate_size=ref_model.config.intermediate_size,
    size_multiplier=None,
    act_fn=ref_model.config.hidden_act,
    gated=True,
    ffw_bias=False,
    ffw_prenorm=True,
    ffw_postnorm=False,
    # ==================
    # Norm Config
    # ==================
    norm_type="RMSNorm",
    norm_eps=ref_model.config.rms_norm_eps,
    # ==================
    # RoPE Config
    # ==================
    rope_theta=ref_model.config.rope_theta,
    rope_pattern="all",
    # ==================
    # Initialization Config
    # ==================
    init_std=ref_model.config.initializer_range,
)
model = Transformer(config)
model

[2025-07-25 16:30:04,179][config][INFO] - Using RoPE for all layers


Transformer(
  (tok_embeddings): Embedding(151936, 1024)
  (layers): ModuleList(
    (0-27): 28 x TransformerBlock(
      (attn_prenorm): RMSNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qnorm): RMSNorm((128,), eps=1e-06, elementwise_affine=True)
        (knorm): RMSNorm((128,), eps=1e-06, elementwise_affine=True)
        (wq): Linear(in_features=1024, out_features=2048, bias=False)
        (wk): Linear(in_features=1024, out_features=1024, bias=False)
        (wv): Linear(in_features=1024, out_features=1024, bias=False)
        (wo): Linear(in_features=2048, out_features=1024, bias=False)
        (sdpa): ScaledDotProductAttention()
      )
      (ffw_prenorm): RMSNorm((1024,), eps=1e-06, elementwise_affine=True)
      (ffw): FeedForward(
        (act_fn): SiLU()
        (wup): Linear(in_features=1024, out_features=3072, bias=False)
        (wdown): Linear(in_features=3072, out_features=1024, bias=False)
        (wgate): Linear(in_features=1024, out

In [52]:
ref_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qwe

In [53]:
def load_qwen3_into_transformer(qwen3_model, transformer_model):
    qwen_state_dict = qwen3_model.state_dict()
    transformer_state_dict = transformer_model.state_dict()
    
    new_state_dict = {}
    
    for name, param in qwen_state_dict.items():
        new_name = name

        # --- Top-level ---
        new_name = new_name.replace("model.embed_tokens.", "tok_embeddings.")
        new_name = new_name.replace("model.norm.", "norm.")
        new_name = new_name.replace("lm_head.", "lm_head.")

        # --- Layers ---
        new_name = new_name.replace("model.layers.", "layers.")

        # --- Attention Projections ---
        new_name = new_name.replace(".self_attn.q_proj.", ".attn.wq.")
        new_name = new_name.replace(".self_attn.k_proj.", ".attn.wk.")
        new_name = new_name.replace(".self_attn.v_proj.", ".attn.wv.")
        new_name = new_name.replace(".self_attn.o_proj.", ".attn.wo.")

        # --- Attention Norms ---
        new_name = new_name.replace(".self_attn.q_norm.", ".attn.qnorm.")
        new_name = new_name.replace(".self_attn.k_norm.", ".attn.knorm.")

        # --- Layer Norms ---
        new_name = new_name.replace(".input_layernorm.", ".attn_prenorm.")
        new_name = new_name.replace(".post_attention_layernorm.", ".ffw_prenorm.")

        # --- FeedForward (MLP) ---
        new_name = new_name.replace(".mlp.gate_proj.", ".ffw.wgate.")
        new_name = new_name.replace(".mlp.up_proj.", ".ffw.wup.")
        new_name = new_name.replace(".mlp.down_proj.", ".ffw.wdown.")

        # Skip rotary embedding
        if "rotary_emb" in name:
            continue

        if new_name in transformer_state_dict:
            if transformer_state_dict[new_name].shape != param.shape:
                print(f"[shape mismatch] {new_name}: {param.shape} vs {transformer_state_dict[new_name].shape}")
                continue
            new_state_dict[new_name] = param
        else:
            print(f"[skipped] No match for: {name} → {new_name}")

    # Load into model
    missing_keys, unexpected_keys = transformer_model.load_state_dict(new_state_dict, strict=False)

    print("\n✅ Loaded parameters.")
    print(f"\t🔸 Missing keys in Transformer: {missing_keys}")
    print(f"\t🔸 Unexpected keys from Qwen3: {unexpected_keys}")

    return transformer_model


In [54]:
model = load_qwen3_into_transformer(ref_model, model)


✅ Loaded parameters.
	🔸 Missing keys in Transformer: ['freqs_cis']
	🔸 Unexpected keys from Qwen3: []


In [55]:
input_ids = torch.randint(0, config.vocab_size, (2, 10))  # Example input
pos_ids = torch.arange(input_ids.shape[1]).unsqueeze(0)  # Create a tensor for positions
seqlen = input_ids.shape[1]

In [56]:
ref_model = ref_model.eval()
model = model.eval()

In [57]:
# with torch.inference_mode():
#     ref_out = ref_model(input_ids=input_ids).logits
# ref_out

# with torch.inference_mode():
#     out = model(input_ids=input_ids)
# out

### Embeddings

In [58]:
with torch.inference_mode():
    ref_emb = ref_model.model.embed_tokens(input_ids) 
    emb = model.tok_embeddings(input_ids)

torch.allclose(ref_emb, emb, atol=1e-5)  # Check if embeddings match

True

### Norms

In [59]:

with torch.inference_mode():
    pos_emb = ref_model.model.rotary_emb(ref_emb, pos_ids)
    ref_att = ref_model.model.layers[0].self_attn
    
    hidden_states = ref_model.model.layers[0].input_layernorm(ref_emb)

    input_shape = hidden_states.shape[:-1]
    hidden_shape = (*input_shape, -1, ref_model.config.head_dim)

    query_states = ref_att.q_norm(ref_att.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
    key_states = ref_att.k_norm(ref_att.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
    value_states = ref_att.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

with torch.inference_mode():
    freqs_cis = model.freqs_cis
    x = model.layers[0].attn_prenorm(ref_emb)
    layer = model.layers[0].attn
    
    bs, seqlen, _ = x.shape
    xq = layer.wq(x)  # (bs, seqlen, n_local_heads * head_dim)
    xk = layer.wk(x)  # (bs, seqlen, n_kv_heads * head_dim)
    xv = layer.wv(x)  # (bs, seqlen, n_kv_heads * head_dim)

    # ==== QK Normalization if applicable
    if layer.use_qknorm and layer.qknorm_use_global:
        # (Option 1) Normalize over the full concatenated dimension first
        xq = layer.qnorm(xq)  # no shape change
        xk = layer.knorm(xk)  # no shape change

    # NOTE: using -1 instead of `n_heads` (or `n_kv_heads`) to infer the actual local heads
    # from sizes of xq, xk, and xv as TP may have sharded them after the above linear ops
    xq = xq.view(bs, seqlen, -1, layer.head_dim)  # (bs, seqlen, n_local_heads, head_dim)
    xk = xk.view(bs, seqlen, -1, layer.head_dim)  # (bs, seqlen, n_kv_heads, head_dim)
    xv = xv.view(bs, seqlen, -1, layer.head_dim)  # (bs, seqlen, n_kv_heads, head_dim)

    if layer.use_qknorm and not layer.qknorm_use_global:
        # (Option 2) Normalise each head independently after reshaping
        xq = layer.qnorm(xq)  # no shape change
        xk = layer.knorm(xk)  # no shape change

In [60]:
# Attention prenorm
torch.allclose(hidden_states, x)

True

In [61]:
# Norms
(
    torch.allclose(query_states, xq.transpose(1, 2)),  
    torch.allclose(key_states, xk.transpose(1, 2)),    
    torch.allclose(value_states, xv.transpose(1, 2)), 
)

(True, True, True)

In [65]:
freqs_cis = model.freqs_cis[0:seqlen]
cos, sin = ref_model.model.rotary_emb(hidden_states, pos_ids)

In [102]:
def reconstruct(freqs_cis, cos, sin):
    
    cos_vals = torch.real(freqs_cis)
    sin_vals = torch.imag(freqs_cis)

    # Duplicate to match the full head_dim, as done in the reference implementation
    cos_vals = torch.cat([cos_vals, cos_vals], dim=-1).unsqueeze(0)
    sin_vals = torch.cat([sin_vals, sin_vals], dim=-1).unsqueeze(0)

    print("Rec:", cos.shape, sin.shape)
    print("Rec:", cos_vals.shape, sin_vals.shape)

    print("Rec:",
        torch.allclose(cos, cos_vals, atol=1e-5),
        torch.allclose(sin, sin_vals, atol=1e-5)
    )

    return cos_vals, sin_vals

_ = reconstruct(freqs_cis, cos, sin)

def reconstruct_reverse(freqs_cis, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # The head_dim is the last dimension
    head_dim = cos.shape[-1]
    
    # Take the first half of the values, as the second half is a duplicate
    cos_half = cos[0, :, :head_dim // 2]
    sin_half = sin[0, :, :head_dim // 2]
    
    # Combine the real (cos) and imaginary (sin) parts to form the complex tensor
    freqs_cis_reconstructed = torch.complex(cos_half, sin_half)

    print(freqs_cis_reconstructed.shape, freqs_cis.shape)
    print(torch.allclose(freqs_cis_reconstructed, freqs_cis, atol=1e-5))
    
    return freqs_cis_reconstructed

_ = reconstruct_reverse(freqs_cis, cos, sin)


Rec: torch.Size([1, 10, 128]) torch.Size([1, 10, 128])
Rec: torch.Size([1, 10, 128]) torch.Size([1, 10, 128])
Rec: True True
torch.Size([10, 64]) torch.Size([10, 64])
True


In [ ]:
from primer.model import apply_rotary_emb
from transformers.models.qwen3.modeling_qwen3 import apply_rotary_pos_emb
from torch import Tensor


def reshape_for_broadcast(freqs_cis: Tensor, x: Tensor) -> Tensor:
    """
    Reshape frequency tensor for broadcasting it with another tensor.
    x is assumed to have shape (bs, seqlen, n_heads, head_dim/2) as a complex tensor.
    freqs_cis is assumed to have shape (max_seqlen, head_dim/2) as a complex tensor.
    """
    ndim = x.ndim
    assert ndim == 4, "Input tensor must have 4 dimensions"
    
    # The sequence length is the second dimension (index 1)
    seqlen = x.shape[1]
    freqs_cis = freqs_cis[:seqlen] # Slice to the correct length

    # Reshape freqs_cis to (1, seqlen, 1, head_dim/2) to broadcast across
    # the batch and head dimensions of x.
    shape = (1, seqlen, 1, x.shape[-1])
    return freqs_cis.view(shape)

def my_apply_rotary_emb(xq: Tensor, xk: Tensor, freqs_cis: Tensor) -> tuple[Tensor, Tensor]:
    """Apply rotary embeddings to input tensors using the given frequency tensor through complex multiplication."""
    
    # Input: (bs, seqlen, n_heads, head_dim)
    # Reshape: (bs, seqlen, n_heads, head_dim/2, 2)
    # View as complex: (bs, seqlen, n_heads, head_dim/2)
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
        
    # freqs_cis is reshaped to (1, seqlen, 1, head_dim/2) for broadcasting
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    
    # Element-wise multiplication:
    # (bs, seqlen, n_heads, dim/2) * (1, seqlen, 1, dim/2)
    # The result is broadcast across the batch and head dimensions.
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(start_dim=3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(start_dim=3)

    return xq_out.type_as(xq), xk_out.type_as(xk)

def my2hf():
    q_, k_ = apply_rotary_pos_emb(query_states, key_states, cos, sin)
    _cos, _sin = reconstruct(freqs_cis, cos, sin)
    q__, k__ = apply_rotary_pos_emb(xq, xk, _cos, _sin, unsqueeze_dim=2)
    q___, k___ = my_apply_rotary_emb(xq, xk, freqs_cis)
    print(
        # torch.allclose(q_, q__.transpose(1, 2), atol=1e-5),
        # torch.allclose(k_, k__.transpose(1, 2), atol=1e-5),
        torch.allclose(q_, q___.transpose(1, 2), atol=1e-5),
        torch.allclose(k_, k___.transpose(1, 2), atol=1e-5),
    )

def hffunc_myinputs():
    q_, k_ = apply_rotary_pos_emb(query_states, key_states, cos, sin)
    
    _c, _s = reconstruct(freqs_cis, cos, sin)
    q__, k__ = apply_rotary_pos_emb(xq, xk, _c, _s, unsqueeze_dim=2) # NOTE dim
    print(
        torch.allclose(q_, q__.transpose(1, 2), atol=1e-5),
        torch.allclose(k_, k__.transpose(1, 2), atol=1e-5)
    )

def myfunc_hfinputs():
    q_, k_ = apply_rotary_emb(xq, xk, freqs_cis)
    _f = reconstruct_reverse(freqs_cis, cos, sin)
    q__, k__ = apply_rotary_emb(xq, xk, _f)
    print(
        torch.allclose(q_, q__, atol=1e-5),
        torch.allclose(k_, k__, atol=1e-5)
    )

hffunc_myinputs()
myfunc_hfinputs()


Rec: torch.Size([1, 10, 128]) torch.Size([1, 10, 128])
Rec: torch.Size([1, 10, 128]) torch.Size([1, 10, 128])
Rec: True True
True True
torch.Size([10, 64]) torch.Size([10, 64])
True
True True


In [ ]:
from transformers.models.qwen3.modeling_qwen3 import apply_rotary_pos_emb


cos, sin = ref_model.model.rotary_emb(hidden_states, pos_ids)
query_states_, key_states_ = apply_rotary_pos_emb(query_states, key_states, cos, sin)

In [ ]:
ref_model.model.rotary_emb(hidden_states, pos_ids)

In [ ]:
def freqs_cis_to_cos_and_sin(freqs_cis):
    """
    Converts complex freqs_cis: (seqlen, head_dim // 2)
    into cos, sin of shape: (1, seqlen, head_dim)
    """
    cos = freqs_cis[:seqlen, ...].real  # (seqlen, head_dim // 2)
    sin = freqs_cis[:seqlen, ...].imag  # (seqlen, head_dim // 2)

    # Interleave real and imag as even-odd positions
    cos_interleaved = torch.zeros(cos.shape[0], cos.shape[1] * 2, device=cos.device)
    sin_interleaved = torch.zeros_like(cos_interleaved)

    cos_interleaved[:, 0::2] = cos
    cos_interleaved[:, 1::2] = cos
    sin_interleaved[:, 0::2] = -sin
    sin_interleaved[:, 1::2] = sin

    return cos_interleaved[None, :, :], sin_interleaved[None, :, :]

c, s = freqs_cis_to_cos_and_sin(freqs_cis)
xq_rot, xk_rot = apply_rotary_pos_emb(xq.transpose(1, 2), xk.transpose(1, 2), c, s)



In [ ]:
torch.allclose(xq_rot , query_states_, 1e-5)

In [ ]:
freqs_cis.shape

In [ ]:
from primer.model import apply_rotary_emb

xq_, xk_ = apply_rotary_emb(xq, xk, freqs_cis=freqs_cis)

In [ ]:
query_states_.shape, xq_.shape

In [ ]:
cos.shape == sin.shape == (1, seqlen, config.head_dim)


In [ ]:
cos

In [ ]:
xq_ = xq.float().reshape(*xq.shape[:-1], -1, 2)

(
    xq.shape, 
    xq_.shape,
    torch.view_as_complex(xq_).shape,
)


In [ ]:
theta, head_dim = config.rope_theta, config.head_dim
freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))
inv_freq = ref_model.model.rotary_emb.inv_freq
torch.allclose(freqs, inv_freq, atol=1e-5)  # Check if frequencies match

In [ ]:
inv_freq_expanded = inv_freq[None, :, None].float().expand(pos_ids.shape[0], -1, 1).to(x.device)
pos_ids_expanded = pos_ids[:, None, :].float()

inv_freq.shape, pos_ids.shape, inv_freq_expanded.shape, pos_ids_expanded.shape

In [ ]:
freqs_ = (inv_freq_expanded.float() @ pos_ids.float()).transpose(1, 2)

In [ ]:
max_seqlen = input_ids.shape[1]

freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))
t = torch.arange(max_seqlen, device=freqs.device)

# The outer product creates a matrix where each row corresponds to a position in the sequence
# and each column corresponds to a frequency component.
freqs = torch.outer(t, freqs).float()  # (max_seqlen, head_dim // 2)


In [ ]:
freqs.shape, freqs_.shape, torch.allclose(freqs, freqs_.squeeze(0), atol=1e-5)

In [ ]:
emb = torch.cat((freqs_, freqs_), dim=-1)
cos_ = emb.cos()
sin_ = emb.sin()

# Extract real (cos) and imaginary (sin) parts
cos = freqs_cis.real  # (max_seqlen, head_dim // 2)
sin = freqs_cis.imag  # (max_seqlen, head_dim // 2)

# Duplicate across last dim to match Qwen3 (concatenating with itself)
# This results in shape: (max_seqlen, head_dim)
cos = torch.cat([cos, cos], dim=-1)
sin = torch.cat([sin, sin], dim=-1)

# Add singleton middle dimension
cos = cos[:, None, :]  # (max_seqlen, 1, head_dim)
sin = sin[:, None, :]  # (max_seqlen, 1, head_dim)

In [ ]:
(
    cos_.shape, 
    sin_.shape, 
    cos.shape, 
    sin.shape, 
    torch.allclose(cos_, cos[:max_seqlen].transpose(0, 1), atol=1e-5), 
    torch.allclose(sin_, sin[:max_seqlen].transpose(0, 1), atol=1e-5)
)

In [ ]:
freqs_cis = model.freqs_cis
xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))  # (bs, seqlen, n_kv_heads, head_dim/2)
print(freqs_cis.shape, xq_.shape)

ndim = xq_.ndim
assert ndim > 1
seqlen = xq_.shape[1]
freqs_cis = freqs_cis[0:seqlen]
print(freqs_cis.shape)
assert freqs_cis.shape == (seqlen, xq_.shape[-1])
shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(xq_.shape)]
freqs_cis = freqs_cis.view(*shape)
print(freqs_cis.shape)


In [ ]:
from torch import Tensor
def reshape_for_broadcast(freqs_cis, x):
    """
    Reshape frequency tensor for broadcasting it with another tensor.
    x is assumed to have shape (bs, n_heads, seqlen, head_dim/2) as a complex tensor.
    freqs_cis is assumed to have shape (max_seqlen, head_dim/2) as a complex tensor.
    """
    ndim = x.ndim
    assert ndim == 4, "Input tensor must have 4 dimensions"
    
    # The sequence length is now the third dimension (index 2)
    seqlen = x.shape[2]
    freqs_cis = freqs_cis[:seqlen] # Slice to the correct length

    # Reshape freqs_cis to (1, 1, seqlen, head_dim/2) to broadcast across
    # the batch and head dimensions of x.
    shape = (1, 1, seqlen, x.shape[-1])
    return freqs_cis.view(shape)

def apply_rotary_emb(xq: Tensor, xk: Tensor, freqs_cis: Tensor) -> tuple[Tensor, Tensor]:
    """Apply rotary embeddings to input tensors using the given frequency tensor through complex multiplication."""
    
    # Input: (bs, seqlen, n_heads, head_dim)
    # Reshape: (bs, seqlen, n_heads, head_dim/2, 2)
    # View as complex: (bs, seqlen, n_heads, head_dim/2)
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))  # (bs, seqlen, n_kv_heads, head_dim/2)
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))  # (bs, seqlen, n_kv_heads, head_dim/2)
        
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)  # (1, seqlen, 1, head_dim/2)
    
    # Input: (bs, seqlen, n_heads, head_dim/2)
    # Multiplication: (bs, seqlen, n_heads, head_dim/2)
    # View as real: (bs, seqlen, n_heads, head_dim/2, 2)
    # Reshape: (bs, seqlen, n_heads, head_dim)
    xq_out = torch.view_as_real(xq_ * freqs_cis).reshape_as(xq)  # (bs, seqlen, n_heads, head_dim)
    xk_out = torch.view_as_real(xk_ * freqs_cis).reshape_as(xk)  # (bs, seqlen, n_kv_heads, head_dim)
    return xq_out.type_as(xq), xk_out.type_as(xk)

In [ ]:
_xq = xq.transpose(1, 2)  # Shape becomes (bs, n_heads, seqlen, head_dim)
_xk = xk.transpose(1, 2)  # Shape becomes (bs, n_kv_heads, seqlen, head_dim)
_xq.shape, freqs_cis.shape

In [ ]:
__xq, __xk = apply_rotary_emb(_xq, _xk, freqs_cis=freqs_cis)

In [ ]:
__xq.shape

In [ ]:
torch.allclose(query_states, __xq, atol=1e-5)

In [ ]:
shape

In [ ]:
freqs_cis.view(*shape).shape

In [ ]:
xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))

xq.shape, xq_.shape, xk.shape, xk_.shape


In [ ]:
ref_model.config.max_position_embeddings, ref_model.config.head_dim, ref_model.config.rope_theta

In [ ]:
config.max_seqlen, config.head_dim, config.rope_theta

In [ ]:
freqs_cis.shape

In [ ]:
xq.shape

In [ ]:
from primer.model import precompute_freqs_cis, apply_rotary_emb
from transformers.models.qwen3.modeling_qwen3 import Qwen3RotaryEmbedding, apply_rotary_pos_emb


freqs_cis = model.freqs_cis
xq_out, xk_out = apply_rotary_emb(xq, xk, freqs_cis)

rotary_emb = Qwen3RotaryEmbedding(ref_model.config)  # or simplified version
cos_, sin_ = rotary_emb(xq, pos_ids)
print(cos_.shape, sin_.shape)

# Extract real (cos) and imaginary (sin) parts
cos = freqs_cis.real  # (max_seqlen, head_dim // 2)
sin = freqs_cis.imag  # (max_seqlen, head_dim // 2)
cos = torch.cat([cos, cos], dim=-1)
sin = torch.cat([sin, sin], dim=-1)
cos = cos[None, :xq.shape[1], ...]  # (head_dim, max_seqlen)
sin = sin[None, :xq.shape[1], ...]  # (head_dim, max_seqlen)
cos = cos.expand(xq.shape[0], -1, -1)  # (batch, seq_len, head_dim)
sin = sin.expand(xq.shape[0], -1, -1)
print(cos.shape, sin.shape)

assert torch.allclose(cos_, cos, atol=1e-5), "Cosine values do not match"
assert torch.allclose(sin_, sin, atol=1e-5), "Sine values do not match"

xq_out_, xk_out_ = apply_rotary_pos_emb(xq.transpose(1, 2), xk.transpose(1, 2), cos, sin)
xq_out_, xk_out_ = xq_out_.transpose(1, 2), xk_out_.transpose(1, 2)

print(xq_out.shape, xk_out.shape, xq_out_.shape, xk_out_.shape)

torch.allclose(xq_out, xq_out_, atol=1e-5), torch.allclose(xk_out, xk_out_, atol=1e-5)

In [ ]:
cos.shape

In [ ]:
xq_out.shape, xk_out.shape, xq_out_.shape, xk_out_.shape

In [ ]:
torch.allclose(xq_out, xq_out_, atol=1e-5), torch.allclose(xk_out, xk_out_, atol=1e-5)

In [ ]:
query_states.shape, key_states.shape, value_states.shape

In [ ]:
cos, sin = pos_emb
freqs_cis = torch.complex(cos[0], sin[0]) 

In [ ]:
freqs_cis.shape, model.freqs_cis.shape

In [ ]:
import torch
import torch.nn as nn

def minimal_rope_test():
    """
    Minimal test to compare your RoPE implementation vs Hugging Face style
    """
    # Test parameters
    batch_size = 1
    seq_len = 4
    n_heads = 2
    head_dim = 8
    theta = 10000.0
    
    print(f"Test config: batch={batch_size}, seq_len={seq_len}, n_heads={n_heads}, head_dim={head_dim}")
    
    # Create test tensors
    torch.manual_seed(42)
    xq = torch.randn(batch_size, seq_len, n_heads, head_dim)
    xk = torch.randn(batch_size, seq_len, n_heads, head_dim)
    
    print(f"Input shapes: xq={xq.shape}, xk={xk.shape}")
    
    # ==================== YOUR IMPLEMENTATION ====================
    def precompute_freqs_cis(dim, end, theta=10000.0):
        freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
        t = torch.arange(end)
        freqs = torch.outer(t, freqs).float()
        freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
        return freqs_cis
    
    def reshape_for_broadcast(freqs_cis, x):
        ndim = x.ndim
        seqlen = x.shape[1]
        freqs_cis = freqs_cis[0:seqlen]
        shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
        return freqs_cis.view(*shape)
    
    def apply_rotary_emb(xq, xk, freqs_cis):
        xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
        xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
        freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
        xq_out = torch.view_as_real(xq_ * freqs_cis).reshape_as(xq)
        xk_out = torch.view_as_real(xk_ * freqs_cis).reshape_as(xk)
        return xq_out.type_as(xq), xk_out.type_as(xk)
    
    # Apply your method
    freqs_cis = precompute_freqs_cis(head_dim, seq_len * 2, theta)
    print(f"freqs_cis shape: {freqs_cis.shape}")
    
    xq_yours, xk_yours = apply_rotary_emb(xq, xk, freqs_cis)
    
    # ==================== HUGGING FACE IMPLEMENTATION ====================
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)
    
    def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
        cos = cos.unsqueeze(unsqueeze_dim)
        sin = sin.unsqueeze(unsqueeze_dim)
        q_embed = (q * cos) + (rotate_half(q) * sin)
        k_embed = (k * cos) + (rotate_half(k) * sin)
        return q_embed, k_embed
    
    def create_hf_cos_sin(head_dim, seq_len, theta=10000.0):
        # Create position IDs
        position_ids = torch.arange(seq_len).unsqueeze(0)  # (1, seq_len)
        
        # Create inverse frequencies
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        
        # Compute frequencies
        inv_freq_expanded = inv_freq[None, :, None].float().expand(1, -1, 1)
        position_ids_expanded = position_ids[:, None, :].float()
        freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
        
        # Create cos/sin embeddings
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos()
        sin = emb.sin()
        
        return cos, sin
    
    # Create HF cos/sin
    cos_hf, sin_hf = create_hf_cos_sin(head_dim, seq_len, theta)
    print(f"HF cos/sin shapes: {cos_hf.shape}, {sin_hf.shape}")
    
    # Apply HF method (need to transpose for their format)
    xq_hf = xq.transpose(1, 2)  # (batch, n_heads, seq_len, head_dim)
    xk_hf = xk.transpose(1, 2)
    
    xq_hf_out, xk_hf_out = apply_rotary_pos_emb(xq_hf, xk_hf, cos_hf, sin_hf)
    
    # Transpose back to your format
    xq_hf_out = xq_hf_out.transpose(1, 2)
    xk_hf_out = xk_hf_out.transpose(1, 2)
    
    # ==================== ALTERNATIVE: CONVERT YOUR freqs_cis TO HF FORMAT ====================
    def convert_freqs_cis_to_hf(freqs_cis, seq_len, batch_size):
        cos_half = freqs_cis[:seq_len].real  # (seq_len, head_dim//2)
        sin_half = freqs_cis[:seq_len].imag  # (seq_len, head_dim//2)
        
        # Duplicate to full head_dim
        cos = torch.cat([cos_half, cos_half], dim=-1)  # (seq_len, head_dim)
        sin = torch.cat([sin_half, sin_half], dim=-1)  # (seq_len, head_dim)
        
        # Add batch dimension
        cos = cos.unsqueeze(0).expand(batch_size, -1, -1)
        sin = sin.unsqueeze(0).expand(batch_size, -1, -1)
        
        return cos, sin
    
    cos_converted, sin_converted = convert_freqs_cis_to_hf(freqs_cis, seq_len, batch_size)
    print(f"Converted cos/sin shapes: {cos_converted.shape}, {sin_converted.shape}")
    
    # Apply with converted cos/sin
    xq_converted_out, xk_converted_out = apply_rotary_pos_emb(xq_hf, xk_hf, cos_converted, sin_converted)
    xq_converted_out = xq_converted_out.transpose(1, 2)
    xk_converted_out = xk_converted_out.transpose(1, 2)
    
    # ==================== COMPARISON ====================
    print("\n" + "="*50)
    print("COMPARISON RESULTS:")
    print("="*50)
    
    print(f"Your method vs HF method:")
    q_match_hf = torch.allclose(xq_yours, xq_hf_out, atol=1e-5)
    k_match_hf = torch.allclose(xk_yours, xk_hf_out, atol=1e-5)
    print(f"  Q matches: {q_match_hf}")
    print(f"  K matches: {k_match_hf}")
    
    if not q_match_hf:
        print(f"  Q max diff: {(xq_yours - xq_hf_out).abs().max().item():.8f}")
    if not k_match_hf:
        print(f"  K max diff: {(xk_yours - xk_hf_out).abs().max().item():.8f}")
    
    print(f"\nYour method vs Converted method:")
    q_match_conv = torch.allclose(xq_yours, xq_converted_out, atol=1e-5)
    k_match_conv = torch.allclose(xk_yours, xk_converted_out, atol=1e-5)
    print(f"  Q matches: {q_match_conv}")
    print(f"  K matches: {k_match_conv}")
    
    if not q_match_conv:
        print(f"  Q max diff: {(xq_yours - xq_converted_out).abs().max().item():.8f}")
    if not k_match_conv:
        print(f"  K max diff: {(xk_yours - xk_converted_out).abs().max().item():.8f}")
    
    print(f"\nHF method vs Converted method:")
    q_match_hf_conv = torch.allclose(xq_hf_out, xq_converted_out, atol=1e-5)
    k_match_hf_conv = torch.allclose(xk_hf_out, xk_converted_out, atol=1e-5)
    print(f"  Q matches: {q_match_hf_conv}")
    print(f"  K matches: {k_match_hf_conv}")
    
    # ==================== DEBUG INFO ====================
    print("\n" + "="*50)
    print("DEBUG INFO:")
    print("="*50)
    
    print("freqs_cis sample values:")
    print(f"  Real part: {freqs_cis[:2, :2].real}")
    print(f"  Imag part: {freqs_cis[:2, :2].imag}")
    
    print("HF cos/sin sample values:")
    print(f"  cos: {cos_hf[0, :2, :4]}")
    print(f"  sin: {sin_hf[0, :2, :4]}")
    
    print("Converted cos/sin sample values:")
    print(f"  cos: {cos_converted[0, :2, :4]}")
    print(f"  sin: {sin_converted[0, :2, :4]}")
    
    # Check if cos/sin values match
    cos_match = torch.allclose(cos_hf, cos_converted, atol=1e-5)
    sin_match = torch.allclose(sin_hf, sin_converted, atol=1e-5)
    print(f"\ncos/sin conversion matches HF: cos={cos_match}, sin={sin_match}")
    
    if not cos_match:
        print(f"cos max diff: {(cos_hf - cos_converted).abs().max().item():.8f}")
    if not sin_match:
        print(f"sin max diff: {(sin_hf - sin_converted).abs().max().item():.8f}")

In [ ]:
minimal_rope_test()

In [ ]:
import torch

def debug_step_by_step():
    """Debug step by step to find the exact issue"""
    
    # Test parameters
    batch_size, seq_len, n_heads, head_dim = 1, 4, 2, 8
    theta = 10000.0
    
    print("=== STEP BY STEP DEBUG ===")
    print(f"Config: batch={batch_size}, seq_len={seq_len}, n_heads={n_heads}, head_dim={head_dim}")
    
    # Create test tensors
    torch.manual_seed(42)
    xq = torch.randn(batch_size, seq_len, n_heads, head_dim)
    xk = torch.randn(batch_size, seq_len, n_heads, head_dim)
    
    print(f"Input tensors: xq={xq.shape}, xk={xk.shape}")
    
    # === YOUR ORIGINAL PRECOMPUTE ===
    def precompute_freqs_cis_original(dim, end, theta=10000.0):
        freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
        t = torch.arange(end, device=freqs.device)  # Make sure same device
        freqs = torch.outer(t, freqs).float()
        freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
        return freqs_cis
    
    freqs_cis = precompute_freqs_cis_original(head_dim, seq_len * 2, theta)
    print(f"freqs_cis shape: {freqs_cis.shape}")  # Should be (8, 4)
    
    # === HF METHOD FOR COMPARISON ===
    def create_hf_cos_sin(dim, seq_len, theta=10000.0):
        position_ids = torch.arange(seq_len).unsqueeze(0)
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        inv_freq_expanded = inv_freq[None, :, None].float().expand(1, -1, 1)
        position_ids_expanded = position_ids[:, None, :].float()
        freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos()
        sin = emb.sin()
        return cos, sin
    
    cos_hf, sin_hf = create_hf_cos_sin(head_dim, seq_len, theta)
    print(f"HF cos/sin shapes: {cos_hf.shape}")
    
    # === CHECK FREQUENCY VALUES ===
    print(f"\nfreqs_cis values at [0:2, 0:2]:")
    print(f"Real: {freqs_cis[:2, :2].real}")
    print(f"Imag: {freqs_cis[:2, :2].imag}")
    
    print(f"\nHF cos values at [0, 0:2, 0:4]: {cos_hf[0, :2, :4]}")
    print(f"HF sin values at [0, 0:2, 0:4]: {sin_hf[0, :2, :4]}")
    
    # === CONVERT YOUR freqs_cis TO HF FORMAT ===
    cos_from_yours = freqs_cis[:seq_len].real  # (seq_len, head_dim//2)
    sin_from_yours = freqs_cis[:seq_len].imag
    cos_from_yours = torch.cat([cos_from_yours, cos_from_yours], dim=-1).unsqueeze(0)
    sin_from_yours = torch.cat([sin_from_yours, sin_from_yours], dim=-1).unsqueeze(0)
    
    print(f"\nConverted from yours cos[0, 0:2, 0:4]: {cos_from_yours[0, :2, :4]}")
    print(f"Converted from yours sin[0, 0:2, 0:4]: {sin_from_yours[0, :2, :4]}")
    
    # Check if they match
    cos_match = torch.allclose(cos_hf, cos_from_yours, atol=1e-6)
    sin_match = torch.allclose(sin_hf, sin_from_yours, atol=1e-6)
    print(f"\nFrequency conversion matches: cos={cos_match}, sin={sin_match}")
    
    if not cos_match:
        print(f"cos diff: {(cos_hf - cos_from_yours).abs().max()}")
    if not sin_match:
        print(f"sin diff: {(sin_hf - sin_from_yours).abs().max()}")
    
    return xq, xk, freqs_cis, cos_hf, sin_hf

def fixed_apply_rotary_emb_final(xq, xk, freqs_cis):
    bsz, seqlen, nheads, dim = xq.shape
    assert dim % 2 == 0
    half_dim = dim // 2

    # freqs_cis shape: (dim, seqlen)
    # transpose to (seqlen, dim)
    freqs_cis = freqs_cis.T  # (seqlen, dim)

    # reshape into pairs: (seqlen, half_dim, 2)
    freqs_cis = freqs_cis.reshape(seqlen, half_dim, 2)  # last dim: real (cos), imag (sin)

    cos = freqs_cis[..., 0]  # (seqlen, half_dim)
    sin = freqs_cis[..., 1]  # (seqlen, half_dim)

    # reshape for broadcasting: (1, seqlen, 1, half_dim)
    cos = cos[None, :, None, :]
    sin = sin[None, :, None, :]

    def apply_rotary(x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        x_rotated_even = x1 * cos - x2 * sin
        x_rotated_odd = x1 * sin + x2 * cos
        x_rotated = torch.stack([x_rotated_even, x_rotated_odd], dim=-1)
        x_rotated = x_rotated.flatten(-2)
        return x_rotated

    return apply_rotary(xq), apply_rotary(xk)

def hf_apply_rotary_pos_emb(q, k, cos, sin):
    """HF style application"""
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)
    
    cos = cos.unsqueeze(1)  # Add head dimension
    sin = sin.unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

def final_test():
    """Final comprehensive test"""
    
    print("=== FINAL TEST ===")
    
    # Get debug data
    xq, xk, freqs_cis, cos_hf, sin_hf = debug_step_by_step()
    
    print(f"\n=== APPLYING METHODS ===")
    
    # Apply your fixed method
    print("Applying YOUR fixed method...")
    xq_yours, xk_yours = fixed_apply_rotary_emb_final(xq, xk, freqs_cis)
    
    # Apply HF method
    print("Applying HF method...")
    xq_hf_input = xq.transpose(1, 2)  # (batch, n_heads, seq_len, head_dim)
    xk_hf_input = xk.transpose(1, 2)
    xq_hf_out, xk_hf_out = hf_apply_rotary_pos_emb(xq_hf_input, xk_hf_input, cos_hf, sin_hf)
    xq_hf_out = xq_hf_out.transpose(1, 2)  # Back to (batch, seq_len, n_heads, head_dim)
    xk_hf_out = xk_hf_out.transpose(1, 2)
    
    print(f"\n=== FINAL COMPARISON ===")
    q_match = torch.allclose(xq_yours, xq_hf_out, atol=1e-5)
    k_match = torch.allclose(xk_yours, xk_hf_out, atol=1e-5)
    
    print(f"Q matches: {q_match}")
    print(f"K matches: {k_match}")
    
    if not q_match:
        diff = (xq_yours - xq_hf_out).abs()
        print(f"Q max diff: {diff.max().item():.8f}")
        print(f"Q mean diff: {diff.mean().item():.8f}")
        # Show where the biggest differences are
        max_idx = diff.argmax()
        coords = torch.unravel_index(max_idx, diff.shape)
        print(f"Max diff location: {coords}")
        print(f"Your value: {xq_yours[coords]}")
        print(f"HF value: {xq_hf_out[coords]}")
        
    if not k_match:
        diff = (xk_yours - xk_hf_out).abs()
        print(f"K max diff: {diff.max().item():.8f}")
        print(f"K mean diff: {diff.mean().item():.8f}")
    
    # Let's also check a few specific values
    print(f"\nSample comparisons:")
    print(f"YOUR Q[0,0,0,:4]: {xq_yours[0,0,0,:4]}")
    print(f"HF   Q[0,0,0,:4]: {xq_hf_out[0,0,0,:4]}")
    print(f"Diff Q[0,0,0,:4]: {(xq_yours[0,0,0,:4] - xq_hf_out[0,0,0,:4]).abs()}")

final_test()